# 🚀 Sistema RAG Completo con Groq (GRATIS y Rápido)
**Arquitectura explicada paso a paso**

**Pasos RAG:**
1. **Carga** → Lee documentos
2. **Divide** → Chunks manejables
3. **Incrusta** → Texto → vectores
4. **Almacena** → Vector DB
5. **Recupera** → Chunks relevantes
6. **Genera** → LLM + contexto

## 1. INSTALACIÓN Y CONFIGURACIÓN

In [1]:
# %pip install langchain-classic langchain-groq langchain-community faiss-cpu python-dotenv langchain-huggingface beautifulsoup4 sentence-transformers

In [2]:
from dotenv import load_dotenv

# Crea un archivo .env en la raíz del proyecto con tu API key:
#   GROQ_API_KEY=gsk_xxxxxxxxxxxx
# Obtén una clave gratuita en: https://console.groq.com/keys
# Para LangSmith (evaluación), añade también:
#   LANGSMITH_API_KEY=lsv2_xxxxxxxxxxxx  → https://smith.langchain.com
#   LANGSMITH_PROJECT=mi-rag-groq
load_dotenv()
print('✅ .env cargado')

✅ .env cargado


## 2. CARGA DE DOCUMENTOS

In [3]:
from langchain_community.document_loaders import WebBaseLoader

# Carga páginas web como documentos. Alternativas habituales:
#   TextLoader('archivo.txt')    → texto plano
#   PyMuPDFLoader('doc.pdf')     → PDFs  (pip install pymupdf)
#   CSVLoader('datos.csv')       → tablas
loader = WebBaseLoader(
    ["https://lilianweng.github.io/posts/2023-06-23-agent/",
     "https://lilianweng.github.io/posts/2025-05-01-thinking/"]
)
docs = loader.load()
print(f"📄 Cargados {len(docs)} documentos")
print(f"Primer doc: {len(docs[0].page_content)} caracteres")

C:\Users\macdu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


📄 Cargados 2 documentos
Primer doc: 43801 caracteres


## 3. DIVISIÓN EN FRAGMENTOS (CHUNKS)

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Divide documentos largos en chunks que quepan en el contexto del LLM.
# chunk_size=1000   → ~750 tokens por chunk (margen de seguridad)
# chunk_overlap=200 → los chunks se solapan 200 chars para no perder
#                     contexto en los cortes (ej: una frase a caballo)
# separators        → corta preferentemente en párrafos, luego líneas,
#                     luego frases; nunca a mitad de palabra
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?"]
)
splits = text_splitter.split_documents(docs)
print(f"✂️ Divididos en {len(splits)} chunks")
print(f"Ejemplo chunk: {len(splits[0].page_content)} chars\n{splits[0].page_content[:200]}...")

✂️ Divididos en 152 chunks
Ejemplo chunk: 637 chars
LLM Powered Autonomous Agents | Lil'Log







































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agen...


## 4. EMBEDDINGS + VECTOR STORE

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# all-MiniLM-L6-v2: modelo de embeddings local (~80 MB), rápido y gratuito.
# Convierte cada chunk en un vector de 384 dimensiones que captura su
# significado semántico. Textos similares → vectores cercanos en el espacio.
# Alternativa multilingüe (mejor para español): 'paraphrase-multilingual-MiniLM-L12-v2'
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# FAISS indexa todos los vectores para hacer búsqueda por similitud coseno
# en ~1ms, independientemente del número de chunks.
# Para producción multi-usuario considera Chroma (local) o Pinecone (nube).
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
print("✅ Vectorstore listo (384-dim vectores indexados)")
print(f"   Chunks indexados: {vectorstore.index.ntotal}")

✅ Vectorstore listo (384-dim vectores indexados)
   Chunks indexados: 152


## 5. LLM + CADENA RAG

In [6]:
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate

# llama-3.3-70b-versatile: mejor modelo del tier gratuito de Groq.
# temperature=0.1 → respuestas deterministas y fieles al contexto.
#                   Sube a 0.7+ solo si quieres respuestas más creativas.
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1
)

# El prompt instruye explícitamente al LLM a responder SOLO con el contexto
# recuperado. Esto es clave para evitar alucinaciones: si la información
# no está en los chunks, el modelo debe decirlo en vez de inventar.
prompt_template = """Usa SOLO información de los documentos proporcionados.
Si no encuentras respuesta, di 'No tengo información suficiente'.

CONTEXTO:
{context}

PREGUNTA: {question}
RESPUESTA CONCISA:"""
PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

# RetrievalQA encadena los 3 pasos automáticamente:
#   1. Recupera los k chunks más relevantes del vectorstore
#   2. Los inyecta en el prompt como {context}
#   3. Llama al LLM y devuelve la respuesta + fuentes
#
# chain_type='stuff': mete todos los chunks en un único prompt.
# Funciona bien con k<=5. Para documentos muy largos usa 'map_reduce'.
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    ),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True  # Devuelve los chunks usados para trazabilidad
)
print("🔥 Cadena RAG lista!")

🔥 Cadena RAG lista!


## 6. ¡PRUEBAS RAG! 🔥

In [7]:
# La búsqueda semántica entiende el significado, no keywords exactos.
# Puedes preguntar en español aunque los documentos estén en inglés;
# all-MiniLM-L6-v2 tiene capacidad cross-lingual básica.
consulta = "¿Qué es la descomposición de tareas en agents de IA?"
# consulta = "¿Cuál es la motivación del artículo Why We Think?"
resultado = qa_chain.invoke({"query": consulta})

print("🤖 RESPUESTA:")
print(resultado["result"])
print("\n" + "="*60)
print("📄 FUENTES (top-2):")
for i, doc in enumerate(resultado["source_documents"][:2]):
    print(f"{i+1}. {doc.metadata.get('source', 'Desconocido')[:60]}")
    print(f"   Preview: {doc.page_content[:200]}...\n")

🤖 RESPUESTA:
La descomposición de tareas en agentes de IA se refiere a la división de un proceso en etapas específicas, que incluyen: 

1. **User Input**: La entrada del usuario.
2. **Task Planning**: La planificación de tareas.
3. **Model Selection**: La selección del modelo.
4. **Task Execution**: La ejecución de la tarea y registro de resultados.

Estas etapas permiten a los agentes de IA procesar y responder a las solicitudes de los usuarios de manera estructurada y eficiente.

📄 FUENTES (top-2):
1. https://lilianweng.github.io/posts/2023-06-23-agent/
   Preview: Prompt LM with 100 most recent observations and to generate 3 most salient high-level questions given a set of observations/statements. Then ask LM to answer those questions.


Planning & Reacting: tr...

2. https://lilianweng.github.io/posts/2023-06-23-agent/
   Preview: Memory stream: is a long-term memory module (external database) that records a comprehensive list of agents’ experience in natural language.

Each elemen

## 7. GUARDAR/RECARGAR VECTORSTORE (PRODUCCIÓN)

In [8]:
# Persiste el índice en disco: genera index.faiss (vectores) + index.pkl (metadatos).
# Así evitas re-embeber los documentos cada vez que reinicias el notebook.
vectorstore.save_local("mi_rag_index")

# allow_dangerous_deserialization=True es requerido porque FAISS usa pickle.
# Solo carga índices que hayas generado tú mismo; nunca de fuentes externas.
vectorstore_reloaded = FAISS.load_local(
    "mi_rag_index", embeddings, allow_dangerous_deserialization=True
)
print("💾 Index guardado y recargado correctamente")
print(f"   Chunks en el índice: {vectorstore_reloaded.index.ntotal}")

💾 Index guardado y recargado correctamente
   Chunks en el índice: 152


## 🎯 PRÓXIMOS PASOS

| Mejora | Cómo | Beneficio |
|--------|------|-----------|
| **Streaming** | `qa_chain.stream()` | Respuestas token a token en tiempo real |
| **Reranking** | `CohereRerank` | Reordena los chunks recuperados → precisión +30% |
| **Cloud DB** | `Chroma` / `Pinecone` | Persistencia escalable y multi-usuario |
| **PDFs** | `PyMuPDFLoader` | Ingesta documentos empresariales |
| **Multi-idioma** | `paraphrase-multilingual-MiniLM-L12-v2` | Embeddings nativos en español y otros idiomas |
| **Evaluación con LangSmith** | `langsmith` + `evaluate()` | Mide la calidad de la cadena RAG con datasets de preguntas/respuestas esperadas. Métricas: relevancia de documentos recuperados, fidelidad de la respuesta al contexto y corrección respecto a la respuesta de referencia. Ver sección 8 ↓ |